# IoT-23 Malicious Traffic Detection

This notebook builds a leakage-resistant binary classifier for IoT-23 Zeek connection logs. It compares a transparent logistic-regression baseline with a dense neural network and evaluates both on captures that were not used for training.

The original university project compared ANN, CNN, LSTM, and Transformer models. Those saved runs are documented in `../RESULTS.md`. This revision does not reuse the old scores because the split and preprocessing methodology have changed.

## Methodology changes

- Entire capture files, rather than random rows, are assigned to train, validation, or test partitions.
- Imputation, encoding, and scaling are fitted only on training data.
- IP addresses, connection IDs, timestamps, capture IDs, and detailed labels are excluded from model inputs.
- Balanced accuracy, precision, recall, F1, ROC AUC, and PR AUC supplement ordinary accuracy.
- A dense network is used because these inputs are unordered tabular features. The old sequence models remain historical experiments.

In [ ]:
from pathlib import Path
import os
import random
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
# reads the Zeek logs IoT-23 ships and converted CSV exports alike
from iot23 import capture_name, find_capture_files, read_connection_log

DATA_DIR = Path(os.environ.get("IOT23_DATA_DIR", REPO_ROOT / "data/iot23")).expanduser()
MAX_ROWS_PER_CAPTURE = int(os.environ.get("MAX_ROWS_PER_CAPTURE", "200000"))
EPOCHS = int(os.environ.get("EPOCHS", "30"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "512"))
SAVE_ARTIFACTS = os.environ.get("SAVE_ARTIFACTS", "0") == "1"

print(f"TensorFlow: {tf.__version__}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Maximum rows per capture: {MAX_ROWS_PER_CAPTURE or 'all'}")

## Load IoT-23 capture files

The notebook reads the Zeek connection logs IoT-23 ships (`*conn.log.labeled`, tab separated with `#` header lines) as well as converted exports (`*conn.log.labeled.csv`, usually pipe separated); `scripts/iot23.py` handles both. Sampling is performed separately within each capture and preserves the benign/malicious ratio when both classes are available. Set `MAX_ROWS_PER_CAPTURE=0` to disable sampling.

In [ ]:
capture_paths = find_capture_files(DATA_DIR)
if len(capture_paths) < 3:
    raise FileNotFoundError(
        f"Found {len(capture_paths)} capture file(s) under {DATA_DIR.resolve()}. "
        "Expected files named *conn.log.labeled or *conn.log.labeled.csv. "
        "At least three are required for disjoint train, validation, and test captures."
    )

pd.DataFrame(
    {"capture_file": [path.name for path in capture_paths], "size_mb": [path.stat().st_size / 1_000_000 for path in capture_paths]}
).round({"size_mb": 1})

In [ ]:
NUMERIC_FEATURES = [
    "id.orig_p", "id.resp_p", "duration", "orig_bytes", "resp_bytes",
    "missed_bytes", "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes",
]
CATEGORICAL_FEATURES = ["proto", "service", "conn_state", "history"]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "label"


def read_capture(path, max_rows=MAX_ROWS_PER_CAPTURE):
    frame = read_connection_log(path)
    if TARGET not in frame.columns:
        raise ValueError(f"{path.name} has no '{TARGET}' column")

    normalized_label = (
        frame[TARGET].astype("string").str.strip().str.split().str[0].str.title()
    )
    valid = normalized_label.isin(["Benign", "Malicious"])
    frame = frame.loc[valid].copy()
    frame[TARGET] = normalized_label.loc[valid].map({"Benign": 0, "Malicious": 1}).astype("int8")
    frame["capture_id"] = capture_name(path)

    for feature in MODEL_FEATURES:
        if feature not in frame.columns:
            frame[feature] = np.nan
    frame = frame[MODEL_FEATURES + [TARGET, "capture_id"]]

    if max_rows > 0 and len(frame) > max_rows:
        counts = frame[TARGET].value_counts()
        stratify = frame[TARGET] if len(counts) == 2 and counts.min() >= 2 else None
        frame, _ = train_test_split(
            frame, train_size=max_rows, random_state=SEED, stratify=stratify
        )
    return frame.reset_index(drop=True)

In [ ]:
frames = []
for path in capture_paths:
    capture = read_capture(path)
    print(f"{path.name}: {len(capture):,} usable rows")
    frames.append(capture)

data = pd.concat(frames, ignore_index=True)
summary = (
    data.groupby(["capture_id", TARGET]).size().unstack(fill_value=0)
    .rename(columns={0: "benign", 1: "malicious"})
)
summary["total"] = summary.sum(axis=1)
print(f"Loaded {len(data):,} rows from {data['capture_id'].nunique()} captures")
summary

In [ ]:
class_counts = data[TARGET].map({0: "Benign", 1: "Malicious"}).value_counts().reindex(["Benign", "Malicious"], fill_value=0)
ax = class_counts.plot.bar(
    color=["#4C78A8", "#E45756"], rot=0
)
ax.set(title="Class distribution after per-capture sampling", xlabel="Label", ylabel="Connections")
plt.tight_layout()
plt.show()

## Split by capture

Rows from one capture can share device, scenario, and collection artifacts. A random row split would expose the test set during training. The helper below searches deterministic group splits until every partition contains both target classes. If the available capture files cannot support that requirement, it stops instead of silently producing an invalid evaluation.

With only a few captures, a holdout measured as a share of rows can cover just one capture; the split then assigns whole captures instead, one to validation and one to testing.


In [ ]:
def grouped_three_way_split(features, labels, groups, seed=SEED):
    if pd.Series(groups).nunique() < 3:
        raise ValueError("At least three distinct capture IDs are required")

    for candidate_seed in range(seed, seed + 500):
        outer = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=candidate_seed)
        train_idx, holdout_idx = next(outer.split(features, labels, groups))
        holdout_groups = groups.iloc[holdout_idx]
        if holdout_groups.nunique() < 2:
            continue

        inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=candidate_seed)
        val_rel, test_rel = next(
            inner.split(features.iloc[holdout_idx], labels.iloc[holdout_idx], holdout_groups)
        )
        val_idx = holdout_idx[val_rel]
        test_idx = holdout_idx[test_rel]
        if all(labels.iloc[idx].nunique() == 2 for idx in (train_idx, val_idx, test_idx)):
            return train_idx, val_idx, test_idx, candidate_seed

    # A 30% holdout measured in rows can only reach one capture when there are few captures of
    # similar size, so fall back to assigning whole captures: one validation, one test, rest train.
    positions = np.arange(len(groups))
    unique_groups = pd.Series(groups).drop_duplicates().tolist()
    for validation_group in unique_groups:
        for test_group in unique_groups:
            if validation_group == test_group:
                continue
            val_mask = (groups == validation_group).to_numpy()
            test_mask = (groups == test_group).to_numpy()
            train_mask = ~(val_mask | test_mask)
            candidate = (positions[train_mask], positions[val_mask], positions[test_mask])
            if all(labels.iloc[idx].nunique() == 2 for idx in candidate):
                return (*candidate, seed)

    raise ValueError(
        "Could not create disjoint capture splits containing both classes. Add more captures "
        "or define a documented capture assignment manually."
    )

X = data[MODEL_FEATURES]
y = data[TARGET]
groups = data["capture_id"]
train_idx, val_idx, test_idx, split_seed = grouped_three_way_split(X, y, groups)

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

split_groups = {
    "train": sorted(groups.iloc[train_idx].unique()),
    "validation": sorted(groups.iloc[val_idx].unique()),
    "test": sorted(groups.iloc[test_idx].unique()),
}
assert set(split_groups["train"]).isdisjoint(split_groups["validation"] + split_groups["test"])
assert set(split_groups["validation"]).isdisjoint(split_groups["test"])

pd.DataFrame(
    {
        "rows": [len(y_train), len(y_val), len(y_test)],
        "malicious_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
        "captures": [", ".join(split_groups[name]) for name in ("train", "validation", "test")],
    },
    index=["train", "validation", "test"],
).assign(split_seed=split_seed)

## Train-only preprocessing

The numeric pipeline fills missing values with training medians and standardizes them. The categorical pipeline fills missing values with the most frequent training value and one-hot encodes categories, including a safe representation for values first encountered in validation or testing.

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=10, dtype=np.float32)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    sparse_threshold=1.0,
)

## Logistic-regression baseline

A deep model should demonstrate value beyond a simpler classifier. Class weighting reduces the tendency to favor the majority label.

In [ ]:
baseline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                solver="saga", max_iter=300, class_weight="balanced", random_state=SEED, n_jobs=-1
            ),
        ),
    ]
)
baseline.fit(X_train, y_train)
baseline_probability = baseline.predict_proba(X_test)[:, 1]

In [ ]:
def evaluate_predictions(model_name, y_true, probability, threshold=0.5):
    prediction = (np.asarray(probability) >= threshold).astype("int8")
    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc": average_precision_score(y_true, probability),
        "threshold": threshold,
    }
    print(pd.Series(metrics).to_string())
    print("\n" + classification_report(y_true, prediction, target_names=["Benign", "Malicious"], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_true, prediction, display_labels=["Benign", "Malicious"], normalize="true", cmap="Blues", values_format=".2f"
    )
    plt.title(f"{model_name}: normalized confusion matrix")
    plt.tight_layout()
    plt.show()
    return metrics

baseline_metrics = evaluate_predictions("Logistic regression", y_test, baseline_probability)

## Dense neural network

The baseline's fitted preprocessor is reused so both models receive the same training-derived representation. Sparse one-hot matrices are converted to dense arrays one batch at a time, avoiding a full dense copy in memory.

In [ ]:
fitted_preprocessor = baseline.named_steps["preprocess"]
X_train_prepared = fitted_preprocessor.transform(X_train)
X_val_prepared = fitted_preprocessor.transform(X_val)
X_test_prepared = fitted_preprocessor.transform(X_test)
print(X_train_prepared.shape, X_val_prepared.shape, X_test_prepared.shape)

In [ ]:
class SparseBatchSequence(tf.keras.utils.Sequence):
    def __init__(self, matrix, labels=None, batch_size=BATCH_SIZE, shuffle=False):
        super().__init__()
        self.matrix = matrix
        self.labels = None if labels is None else np.asarray(labels, dtype=np.float32)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(matrix.shape[0])
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, batch_index):
        rows = self.indices[batch_index * self.batch_size:(batch_index + 1) * self.batch_size]
        batch = self.matrix[rows]
        if sparse.issparse(batch):
            batch = batch.toarray()
        batch = np.asarray(batch, dtype=np.float32)
        if self.labels is None:
            return batch
        return batch, self.labels[rows]

    def on_epoch_end(self):
        if self.shuffle:
            np.random.default_rng(SEED).shuffle(self.indices)

train_batches = SparseBatchSequence(X_train_prepared, y_train, shuffle=True)
validation_batches = SparseBatchSequence(X_val_prepared, y_val)
test_batches = SparseBatchSequence(X_test_prepared)

In [ ]:
classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))

inputs = tf.keras.Input(shape=(X_train_prepared.shape[1],), name="connection_features")
x = tf.keras.layers.Dense(128, activation="relu")(inputs)
x = tf.keras.layers.Dropout(0.30)(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dropout(0.20)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="malicious_probability")(x)
ann = tf.keras.Model(inputs=inputs, outputs=outputs, name="iot23_ann")
ann.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
    ],
)
ann.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-5),
]
history = ann.fit(
    train_batches,
    validation_data=validation_batches,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
history_frame = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_frame[["loss", "val_loss"]].plot(ax=axes[0], title="Training and validation loss")
history_frame[["pr_auc", "val_pr_auc"]].plot(ax=axes[1], title="Training and validation PR AUC")
for axis in axes:
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
ann_probability = ann.predict(test_batches, verbose=1).ravel()
ann_metrics = evaluate_predictions("Dense neural network", y_test, ann_probability)
comparison = pd.DataFrame([baseline_metrics, ann_metrics]).set_index("model")
comparison.round(4)

## Interpretation checklist

Before publishing new numbers:

1. Confirm that capture IDs do not overlap between splits.
2. Report the malicious recall and PR AUC alongside accuracy.
3. Compare the neural network with the logistic baseline rather than assuming deep learning wins.
4. Inspect false negatives and consider threshold selection on the validation set.
5. Record sampling limits and package versions.

A result on unseen captures is more informative than the near-perfect single-capture score from the original coursework.

In [ ]:
if SAVE_ARTIFACTS:
    artifact_dir = REPO_ROOT / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(baseline, artifact_dir / "logistic-baseline.joblib")
    joblib.dump(fitted_preprocessor, artifact_dir / "preprocessor.joblib")
    ann.save(artifact_dir / "iot23-ann.keras")
    comparison.to_csv(artifact_dir / "test-metrics.csv")
    pd.Series(split_groups).to_json(artifact_dir / "capture-splits.json", indent=2)
    print(f"Saved generated artifacts to {artifact_dir.resolve()}")
else:
    print("Set SAVE_ARTIFACTS=1 to save models, preprocessing, metrics, and capture assignments.")

## Limitations and safe use

IoT-23 captures represent specific devices, malware families, collection periods, and labeling decisions. Performance may not transfer to a different network. Flow classification should support, not replace, layered monitoring and analyst review. Only analyze traffic you own or are authorized to inspect. This notebook reads saved Zeek logs and does not execute malware.